## 2.2 Tokenizing Text

In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [6]:
len(raw_text)

20479

In [9]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [35]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [23]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.3 Converting tokens into token IDS

In [31]:
# create sorted "set" of every word - set removes duplicates

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [36]:
# for each word in all_words, assign it an integer and store it in vocab

vocab = {token:integer for integer, token in enumerate(all_words)}
print(len(vocab))

1130


In [61]:
class SimpleTokenizerV1:
    # str_to_int is our Word to ID set
    # int_to_str is our ID to Word set
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    # encode recieves some text, splits it based on previous preprocessing rules
    # then refers to our str_to_int to get the IDs for each word and return the IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # decode recieves a list of ids rather than a list of words
    # then refesrs to our int_to_str to get the words that correspond to each ID
    # joins these back together, adds spaces, and returns the decoded text
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuatinos
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [62]:
# create a tokenizer object using our vocab from "the-verdict.txt"
tokenizer = SimpleTokenizerV1(vocab)

In [63]:
# this is the token representation of the below text
# note this example ONLY contains words that we have captured from "the-verdict.txt" in our vocab

text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [50]:
# going the opposite direction from ids to text
# note there are some slight differences - our decode method isn't perfect

print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [67]:
## this will result in a keyError as the word "Hello" is not in our vocab

text = "Hello, do you like tea?"
#print(tokenizer.encode(text))

## 2.4 Adding special context tokens

In [58]:
# add special context tokens "endoftext" and "unk" to the token set
# note we have to convert it to a list to get the extend functionality

all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

# same enumeration method from before
vocab = {token:integer for integer, token in enumerate(all_tokens)}

1132

In [ ]:
len(vocab.items())

In [60]:
# the new special context tokens are included at the end of our vocab

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [64]:
class SimpleTokenizerV2:
    # str_to_int is our Word to ID set
    # int_to_str is our ID to Word set
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    # encode recieves some text, splits it based on previous preprocessing rules
    # then refers to our str_to_int to get the IDs for each word and return the IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # check if the word exists in vocab, if not assign it the unknown token
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # decode recieves a list of ids rather than a list of words
    # then refesrs to our int_to_str to get the words that correspond to each ID
    # joins these back together, adds spaces, and returns the decoded text
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuatinos
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [65]:
tokenizer = SimpleTokenizerV2(vocab)

In [70]:
# now the decoded version of text has the unknown token rather than throwing a keyError
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea?'

## 2.5 Byte pair encoding

In [71]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


In [76]:
# tiktoken implements BPE (byte pair encoding)
# it work very similarly to our basic example SimpleTokenizer class

tokenizer = tiktoken.get_encoding("gpt2")

In [86]:
# exercise 2.1
# this tokenizer can encode and decode unknown words
# instead of an unknown token - it breaks down unknown words into smaller parts or even individual characters
# this way the output can contain the unknown word after being decoded

integers = tokenizer.encode(text)
print("One word broken into small tokens:", integers)

text = "Akwirw ier"
print("Can still reproduce the original input:", tokenizer.decode(tokenizer.encode(text)))

One word broken into small tokens: [33901, 86, 343, 86, 220, 959]
Can still reproduce the original input: Akwirw ier


In [90]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

# this results in an error as we need to specify the special tokens allowed
# tokenizer.encode(text)

print(tokenizer.encode(text, allowed_special={"<|endoftext|>"}))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


## 2.6 Data sampling with a sliding window

In [92]:
# read in "the-verdict.txt" and encode it with the tiktoken tokenizer

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [93]:
enc_sample = enc_text[50:]

In [101]:
context_size = 4

# first four inputs as specified by the context size
x = enc_sample[:context_size]

# the targets are inputs shifted by one position
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [103]:
# context here is the data the LLM has
# desired is the next value that it will predict

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    #print(context, "---->", desired)
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [107]:
# pytorch implements an efficient dataloader
# there is no need to reinvent the wheel here and do it ourselves
import torch

In [108]:
torch.__version__

'2.5.1+cpu'

In [109]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
    
        # tokenize the entire text
        token_ids = tokenizer.encode(txt)
    
        # use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [111]:
def create_dataloader_v1(txt, batch_size=2, max_length=256,
                        stride=128, shuffle=True, drop_last=True,
                        num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    
    return dataloader

In [112]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [119]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [120]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [121]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings

In [122]:
input_ids = torch.tensor([ 2,   3,    5,   1])

In [129]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, output_dim)

In [130]:
# for each input a 3 dimension random weight is provided
# input in this case is the length of the vocab - three random values for each vocab value
# right now they are random - later during optimization they will be trained and adjusted
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        ...,
        [-0.5931,  1.0895, -0.6854],
        [ 0.7447,  0.5803, -0.4246],
        [-0.3130,  0.7558, -1.2656]], requires_grad=True)


In [126]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [127]:
embedding_layer(torch.tensor([2]))

tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)

In [128]:
embedding_layer(input_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

## 2.8 Encoding word positions

In [131]:
# use the actual sizes for our vocab and tokenizer
# each token has 256 random weights assigned
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [132]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [133]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [134]:
token_embeddings = token_embedding_layer(inputs)

# Batch size of 8, max length (tokens) of 4, 256 weight values
token_embeddings.shape

torch.Size([8, 4, 256])

In [136]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [137]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])
